484 mins 39.0 secs runtime

## Load libraries

In [1]:
import os
import json
import csv

import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL

from sklearn.preprocessing import StandardScaler          # keep CPU scaler (still leak-free)
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm

# NEW: cuML GPU Random Forest
from cuml.ensemble import RandomForestRegressor as cuRFRegressor

## Resume logic

In [2]:
RESULTS_CSV = "rf_cuml_fold_results.csv"

def append_fold_result(
    model_type,
    fold_no,
    lag_set,
    params,
    mae_val,
    rmse_val,
    smape_val,
    mase_val,
):
    """
    Append a single (fold, lag_set, params) result to a CSV on disk.
    Safe against crashes: each call writes one row immediately.
    """
    row = {
        "model_type": model_type,
        "fold_no": fold_no,
        "lag_set": json.dumps(lag_set),      # serialize list
        "params": json.dumps(params, sort_keys=True),  # serialize dict
        "MAE": float(mae_val),
        "RMSE": float(rmse_val),
        "sMAPE": float(smape_val),
        "MASE": float(mase_val),
    }

    file_exists = os.path.exists(RESULTS_CSV)

    with open(RESULTS_CSV, mode="a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

        
def make_lag_key(lag_set):
    """
    Stable string key for a lag_set (list of ints).
    Using sorted so [1,12] and [12,1] map to the same key.
    """
    return json.dumps(sorted(lag_set))

def make_params_key(params):
    """
    Stable string key for params dict.
    """
    return json.dumps(params, sort_keys=True)


def load_completed_runs():
    """
    Read existing results CSV (if present) and return a set of
    (fold_no, lag_key, params_key) that have already been run.
    """
    completed = set()
    if not os.path.exists(RESULTS_CSV):
        return completed

    import pandas as pd
    df_done = pd.read_csv(RESULTS_CSV)

    # If old file without lag_key/params_key, reconstruct them
    if "lag_key" not in df_done.columns:
        df_done["lag_key"] = df_done["lag_set"].apply(lambda s: make_lag_key(json.loads(s)))
    if "params_key" not in df_done.columns:
        df_done["params_key"] = df_done["params"].apply(lambda s: make_params_key(json.loads(s)))

    for _, row in df_done.iterrows():
        completed.add(
            (
                int(row["fold_no"]),
                row["lag_key"],
                row["params_key"],
            )
        )
    return completed


def append_fold_result(
    model_type,
    fold_no,
    lag_set,
    params,
    mae_val,
    rmse_val,
    smape_val,
    mase_val,
):
    """
    Append a single (fold, lag_set, params) result to a CSV on disk.
    Called immediately after computing metrics so work is never lost.
    """
    lag_key = make_lag_key(lag_set)
    params_key = make_params_key(params)

    row = {
        "model_type": model_type,
        "fold_no": fold_no,
        "lag_set": json.dumps(lag_set),
        "params": json.dumps(params, sort_keys=True),
        "lag_key": lag_key,
        "params_key": params_key,
        "MAE": float(mae_val),
        "RMSE": float(rmse_val),
        "sMAPE": float(smape_val),
        "MASE": float(mase_val),
    }

    file_exists = os.path.exists(RESULTS_CSV)

    with open(RESULTS_CSV, mode="a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


## Config

In [3]:

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    "AverageNeighbourPrice",
    "local_I",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
]

categorical_cols = [
    "LMIQuadrant__2",
    "LMIQuadrant__3",
    "LMIQuadrant__4",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# ---- lag plan (always 1 & 12; variants with 2–6; optional 24) ----
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

# RF tuning grid
rf_param_grid = {
    "n_estimators": [250, 500],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5],
    "bootstrap": [True],
}


## Evaluation metric functions

In [4]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale





## Load data

In [5]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL])

mask_tv = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask_tv].copy()

# all unique months in train+val window
dates = sorted(df_tv[TIME_COL].unique())

## Training with STL + Rolling CV

In [6]:
# =========================================================
# CV STRUCTURE:
#   outer loop   = folds (STL + all lags computed once per fold)
#   mid loop     = lag_set (subselect lag columns, scale, etc.)
#   inner loop   = RF params (fit, predict, metrics)
# =========================================================
from collections import defaultdict

# metrics_store[(lag_tuple, params_key)] = dict of metric lists across folds
metrics_store = {}

# Pre-build list of folds based on dates
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > len(dates):
        break

    train_start = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end   = dates[train_end_idx - 1]
    val_start   = dates[val_start_idx]
    val_end     = dates[val_end_idx - 1]

    fold_specs.append((train_start, train_end, val_start, val_end))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

completed_runs = load_completed_runs()
print(f"Found {len(completed_runs)} previously completed (fold, lag, params) runs.")

# =========================================================
# MAIN LOOP: FOLDS → (lag_set → params)
# =========================================================
for fold_no, (train_start, train_end, val_start, val_end) in enumerate(fold_specs, start=1):
    print(f"\n=== Fold {fold_no}: "
          f"Train {train_start:%Y-%m}–{train_end:%Y-%m}, "
          f"Val {val_start:%Y-%m}–{val_end:%Y-%m} ===")

    # ---- slice fold train/val ----
    mask_train = (df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)
    mask_val   = (df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)

    fold_train = df_tv.loc[mask_train].copy()
    fold_val   = df_tv.loc[mask_val].copy()

    # ---- STL on training ONLY (per LA) ----
    fold_train["stl_trend"]    = np.nan
    fold_train["stl_seasonal"] = np.nan
    fold_train["stl_resid"]    = np.nan

    for la, sub in tqdm(fold_train.groupby(ENTITY_COL),
                        desc=f"  STL (train only), fold {fold_no}", leave=False):
        sub = sub.sort_values(TIME_COL)
        series = sub[TARGET_COL].astype(float)
        if len(series) < 24:  # too short for STL, skip
            continue
        stl = STL(series, period=12, robust=True)
        res = stl.fit()
        fold_train.loc[sub.index, "stl_trend"]    = res.trend
        fold_train.loc[sub.index, "stl_seasonal"] = res.seasonal
        fold_train.loc[sub.index, "stl_resid"]    = res.resid

    # ---- extend STL into validation (per LA) ----
    fold_val["stl_trend"]    = np.nan
    fold_val["stl_seasonal"] = np.nan
    fold_val["stl_resid"]    = 0.0  # unknown future residuals

    for la in fold_train[ENTITY_COL].unique():
        sub_train = fold_train[fold_train[ENTITY_COL] == la].sort_values(TIME_COL)
        sub_val   = fold_val[fold_val[ENTITY_COL] == la].sort_values(TIME_COL)

        if sub_val.empty or sub_train["stl_trend"].isna().all():
            continue

        n_future = len(sub_val)

        # seasonal: repeat last 12-month pattern (or fewer if early)
        season_train = sub_train["stl_seasonal"].dropna().values
        if season_train.size == 0:
            continue
        if len(season_train) >= 12:
            base_pattern = season_train[-12:]
        else:
            base_pattern = season_train
        reps = int(np.ceil(n_future / len(base_pattern)))
        season_future = np.tile(base_pattern, reps)[:n_future]

        # trend: simple linear extrapolation
        trend_train = sub_train["stl_trend"].dropna().values
        t_idx = np.arange(len(trend_train))
        if len(trend_train) >= 2:
            coef = np.polyfit(t_idx, trend_train, 1)
            future_t = np.arange(len(trend_train), len(trend_train) + n_future)
            trend_future = coef[0] * future_t + coef[1]
        else:
            trend_future = np.full(n_future, trend_train[-1])

        future_idx = sub_val.index
        fold_val.loc[future_idx, "stl_trend"]    = trend_future
        fold_val.loc[future_idx, "stl_seasonal"] = season_future

    # ---- combine train+val to compute ALL LAG COLS (all_lags) once per fold ----
    fold_train["is_train"] = True
    fold_val["is_train"]   = False

    combined = pd.concat([fold_train, fold_val], axis=0)
    combined = combined.sort_values([ENTITY_COL, TIME_COL])

    # create lag columns for ALL lags (superset) on STL components
    for lag in all_lags:
        for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
            col = f"{comp}_lag{lag}"
            combined[col] = combined.groupby(ENTITY_COL)[comp].shift(lag)

    # ---------------------------------------------------------
    # For this fold: loop over lag_set, then RF params
    # Using the precomputed lag columns in `combined`
    # ---------------------------------------------------------
    for lag_set in lag_combinations:
        print(f"  Lag set: {lag_set}")
        lag_cols = [
            f"{comp}_lag{lag}"
            for comp in ["stl_trend", "stl_seasonal", "stl_resid"]
            for lag in lag_set
        ]

        # split back into train/val
        fold_train_lag = combined[combined["is_train"]].copy()
        fold_val_lag   = combined[~combined["is_train"]].copy()

        # require all lags present
        fold_train_lag = fold_train_lag.dropna(subset=lag_cols)
        fold_val_lag   = fold_val_lag.dropna(subset=lag_cols)

        if fold_train_lag.empty or fold_val_lag.empty:
            print("    (skip: no data after lag drop)")
            continue

        # scale continuous + lag features ON TRAIN ONLY (still leak-free)
        scale_cols = continuous_cols + lag_cols
        scaler = StandardScaler()
        fold_train_lag[scale_cols] = scaler.fit_transform(fold_train_lag[scale_cols])
        fold_val_lag[scale_cols]   = scaler.transform(fold_val_lag[scale_cols])

        # === NEW: convert to float32 numpy arrays for cuML ===
        feature_cols = continuous_cols + categorical_cols + lag_cols

        X_train = fold_train_lag[feature_cols].to_numpy(dtype=np.float32)
        y_train = fold_train_lag[TARGET_COL].to_numpy(dtype=np.float32)

        X_val   = fold_val_lag[feature_cols].to_numpy(dtype=np.float32)
        y_val   = fold_val_lag[TARGET_COL].to_numpy(dtype=np.float32)

        # precompute lag_key for this lag_set
        cur_lag_key = make_lag_key(lag_set)

        # inner loop over RF hyperparameters
        for params in ParameterGrid(rf_param_grid):
            print(f"  RF params: {params}")

            # === RESUME CHECK ===
            cur_params_key = make_params_key(params)
            resume_key = (fold_no, cur_lag_key, cur_params_key)
            if resume_key in completed_runs:
                print("    (skip: already completed, resuming)")
                continue

            lag_key = tuple(lag_set)
            params_key = tuple(sorted(params.items()))
            key = (lag_key, params_key)

            if key not in metrics_store:
                metrics_store[key] = {
                    "lag_set": lag_key,
                    "params": params,
                    "mae": [],
                    "rmse": [],
                    "smape": [],
                    "mase": [],
                    "folds": 0,
                }

            # === NEW: cuML GPU Random Forest ===
            # cuML RF supports n_estimators, max_depth, max_features,
            # min_samples_split, min_samples_leaf, bootstrap, random_state, etc. :contentReference[oaicite:0]{index=0}
            cuml_params = params.copy()
            if cuml_params.get("max_depth") is None:
                # drop it so cuML uses its own default max_depth
                cuml_params.pop("max_depth")

            rf = cuRFRegressor(
                **cuml_params,
                random_state=42,
                n_streams=8,
                output_type="numpy",
            )

            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)


            mae_val   = mae(y_val, y_pred)
            rmse_val  = rmse(y_val, y_pred)
            smape_val = smape(y_val, y_pred)
            mase_val  = mase(y_val, y_pred, y_train)

            metrics_store[key]["mae"].append(mae_val)
            metrics_store[key]["rmse"].append(rmse_val)
            metrics_store[key]["smape"].append(smape_val)
            metrics_store[key]["mase"].append(mase_val)
            metrics_store[key]["folds"] += 1

# NEW: persist this fold’s result to disk
            append_fold_result(
                model_type="RandomForest_cuML",
                fold_no=fold_no,
                lag_set=lag_set,
                params=params,
                mae_val=mae_val,
                rmse_val=rmse_val,
                smape_val=smape_val,
                mase_val=mase_val,
            )





Number of folds: 5
Found 329 previously completed (fold, lag, params) runs.

=== Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03 ===


  Lag set: [1, 12]
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 250}
    (skip: already completed, resuming)
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
    (skip: already completed, resuming)
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 250}
    (skip: already completed, resuming)
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500}
    (skip: already completed, resuming)
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 250}
    (skip: already completed, resuming)
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt

  Lag set: [1, 12]
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estim

  Lag set: [1, 12]
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estim

  Lag set: [1, 12]
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estim

  Lag set: [1, 12]
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 250}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
  RF params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estim

## Results

In [7]:
df_results = pd.read_csv(RESULTS_CSV)

# Convert JSON strings back to Python objects if you want
df_results["lag_set"] = df_results["lag_set"].apply(json.loads)
df_results["params"] = df_results["params"].apply(json.loads)

# Group by lag_set + params string to aggregate over folds
# (we’ll use the JSON-encoded params as a stable key)
df_results["lag_key"] = df_results["lag_set"].apply(lambda x: json.dumps(x))
df_results["params_key"] = df_results["params"].apply(lambda d: json.dumps(d, sort_keys=True))

agg = (
    df_results.groupby(["model_type", "lag_key", "params_key"])
      .agg(
          folds=("fold_no", "nunique"),
          MAE_mean=("MAE", "mean"),
          MAE_std=("MAE", "std"),
          RMSE_mean=("RMSE", "mean"),
          RMSE_std=("RMSE", "std"),
          sMAPE_mean=("sMAPE", "mean"),
          sMAPE_std=("sMAPE", "std"),
          MASE_mean=("MASE", "mean"),
          MASE_std=("MASE", "std"),
      )
      .reset_index()
)

# If you want to restore original lag_set / params objects:
agg["lag_set"] = agg["lag_key"].apply(json.loads)
agg["params"]  = agg["params_key"].apply(json.loads)

results_df = agg.sort_values("RMSE_mean").reset_index(drop=True)
print(results_df.head(20))
results_df.to_csv("rf_cuml_aggregated_results.csv", index=False)

           model_type      lag_key  \
0   RandomForest_cuML  [1, 12, 24]   
1   RandomForest_cuML  [1, 12, 24]   
2   RandomForest_cuML  [1, 12, 24]   
3   RandomForest_cuML  [1, 12, 24]   
4   RandomForest_cuML  [1, 12, 24]   
5   RandomForest_cuML  [1, 12, 24]   
6   RandomForest_cuML  [1, 12, 24]   
7   RandomForest_cuML  [1, 12, 24]   
8   RandomForest_cuML  [1, 12, 24]   
9   RandomForest_cuML  [1, 12, 24]   
10  RandomForest_cuML  [1, 12, 24]   
11  RandomForest_cuML  [1, 12, 24]   
12  RandomForest_cuML  [1, 12, 24]   
13  RandomForest_cuML  [1, 12, 24]   
14  RandomForest_cuML  [1, 12, 24]   
15  RandomForest_cuML  [1, 12, 24]   
16  RandomForest_cuML  [1, 12, 24]   
17  RandomForest_cuML  [1, 12, 24]   
18  RandomForest_cuML  [1, 12, 24]   
19  RandomForest_cuML  [1, 12, 24]   

                                           params_key  folds      MAE_mean  \
0   {"bootstrap": true, "max_depth": 10, "max_feat...      5  11729.942773   
1   {"bootstrap": true, "max_depth": null, "m

In [8]:
print(results_df.head(20))

           model_type      lag_key  \
0   RandomForest_cuML  [1, 12, 24]   
1   RandomForest_cuML  [1, 12, 24]   
2   RandomForest_cuML  [1, 12, 24]   
3   RandomForest_cuML  [1, 12, 24]   
4   RandomForest_cuML  [1, 12, 24]   
5   RandomForest_cuML  [1, 12, 24]   
6   RandomForest_cuML  [1, 12, 24]   
7   RandomForest_cuML  [1, 12, 24]   
8   RandomForest_cuML  [1, 12, 24]   
9   RandomForest_cuML  [1, 12, 24]   
10  RandomForest_cuML  [1, 12, 24]   
11  RandomForest_cuML  [1, 12, 24]   
12  RandomForest_cuML  [1, 12, 24]   
13  RandomForest_cuML  [1, 12, 24]   
14  RandomForest_cuML  [1, 12, 24]   
15  RandomForest_cuML  [1, 12, 24]   
16  RandomForest_cuML  [1, 12, 24]   
17  RandomForest_cuML  [1, 12, 24]   
18  RandomForest_cuML  [1, 12, 24]   
19  RandomForest_cuML  [1, 12, 24]   

                                           params_key  folds      MAE_mean  \
0   {"bootstrap": true, "max_depth": 10, "max_feat...      5  11729.942773   
1   {"bootstrap": true, "max_depth": null, "m